# LAB 4 — Pipeline Naive RAG Completo (Ollama + BGE-M3 + OpenSearch/FAISS)

## Objetivo

Implementar um **RAG pipeline ponta-a-ponta** sobre a infraestrutura provisionada na **Aula 1**:

1. **Docling** — Ingestão de PDFs jurídicos com layout complexo
2. **RecursiveCharacterTextSplitter** — Chunking jurídico com separadores `Art.`, `§`, `Inciso`
3. **BGE-M3 via Ollama** — Embeddings multilíngues (1024 dims) servidos por `ollama serve` em `localhost:11434`
4. **OpenSearch kNN** (servidor da Aula 1) com *fallback* **FAISS** local
5. **Llama 3.2 3B Instruct via Ollama** — LLM servido por `ollama serve` em `localhost:11434`
6. **LangChain LCEL** — Orquestração modular do pipeline

## Stack (idêntica à instalada na Aula 1)

| Camada | Ferramenta | Endpoint |
|---|---|---|
| Servidor LLM | **Ollama** (`llama3.2:3b`) | `http://localhost:11434` |
| Servidor de Embeddings | **Ollama** (`bge-m3`) | `http://localhost:11434` |
| Vector Store | **OpenSearch 3.x** (Podman/Docker da Aula 1) | `http://localhost:9200` |
| Vector Store *(fallback)* | **FAISS** local | — |
| Framework | **LangChain LCEL** + `langchain-ollama` | — |
| Ambiente | Python 3.11 + venv `venv_rag` da Aula 1 | Jupyter/VS Code |

## Pré-requisitos da Aula 1 (verifique antes de prosseguir)

```bash
# Ollama deve estar respondendo:
curl -s http://localhost:11434/api/tags

# Modelos necessários instalados:
ollama pull llama3.2:3b
ollama pull bge-m3        # se não tiver, o notebook cai para FAISS+HuggingFace automaticamente

# OpenSearch (opcional — há fallback FAISS):
curl -s http://localhost:9200/_cluster/health
```

## Diagrama do Pipeline

```
PDFs → Docling → Markdown → Chunking → Ollama(bge-m3) → OpenSearch/FAISS → Retriever → Ollama(llama3.2:3b) → Resposta
                                          (1024 dims)        (top-k=5)              (com citação de fontes)
```


In [ ]:
# Instalação de dependências (compatíveis com o venv_rag da Aula 1)
# Se você já executou o LAB1 da Aula 1, a maioria já está instalada.
import subprocess, sys

packages = [
    'langchain>=0.3', 'langchain-community>=0.3', 'langchain-core>=0.3',
    'langchain-text-splitters>=0.3', 'langchain-ollama>=0.2',
    'sentence-transformers>=3.0',   # fallback para BGE-M3 via HuggingFace
    'faiss-cpu>=1.8', 'opensearch-py>=2.7',
    'docling>=2.0', 'pypdf>=4.0',
    'pandas>=2.2', 'numpy>=1.26', 'matplotlib>=3.9', 'seaborn>=0.13',
    'python-dotenv>=1.0', 'requests>=2.32',
]

print('📦 Instalando dependências...')
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('✅ Instalação completa!\n')

# ─── Verificar que o Ollama da Aula 1 está acessível ───
import os, requests
from dotenv import load_dotenv
load_dotenv()  # carrega ~/mba-rag/.env se existir

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_LLM_MODEL = os.getenv('OLLAMA_LLM_MODEL', 'llama3.2:3b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')

try:
    r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
    r.raise_for_status()
    modelos = [m['name'] for m in r.json().get('models', [])]
    print(f'✅ Ollama OK em {OLLAMA_BASE_URL}')
    print(f'   Modelos instalados: {modelos}')
    if OLLAMA_LLM_MODEL not in modelos:
        print(f'   ⚠️  Modelo LLM "{OLLAMA_LLM_MODEL}" não encontrado. Execute: ollama pull {OLLAMA_LLM_MODEL}')
    if OLLAMA_EMBED_MODEL not in modelos:
        print(f'   ⚠️  Modelo de embedding "{OLLAMA_EMBED_MODEL}" não encontrado. Execute: ollama pull {OLLAMA_EMBED_MODEL}')
        print(f'       (o notebook cai para HuggingFaceEmbeddings("BAAI/bge-m3") se preferir)')
except Exception as e:
    print(f'❌ Ollama não está respondendo em {OLLAMA_BASE_URL}')
    print(f'   Erro: {e}')
    print(f'   Inicie o servidor com `ollama serve` (ou serviço do SO) e refaça esta célula.')


In [ ]:
# Corpus jurídico — duas opções:
#   (A) Corpus INLINE (default, rápido) — strings Python com Lei 11.343, Acórdão HC, etc.
#   (B) Corpus via DOCLING + PDFs REAIS do dataset (Manual_DPCA_atualizado.pdf, Laudo.pdf)
#       Use esta opção para ver a etapa de ingestão Docling no pipeline ponta-a-ponta.
import os
from pathlib import Path

# >>> MUDE AQUI para alternar entre as duas opções:
USE_DOCLING_REAL = False    # True = ingere os PDFs reais via Docling (~30s–3min com OCR)

DATASET_DIR = Path("../datasets").resolve()
PDF_MANUAL  = DATASET_DIR / "Manual_DPCA_atualizado.pdf"
PDF_LAUDO   = DATASET_DIR / "Laudo.pdf"

if USE_DOCLING_REAL and PDF_MANUAL.exists() and PDF_LAUDO.exists():
    # ────────────────────────────────────────────────────────────
    # OPÇÃO B — Docling + PDFs reais do dataset
    # ────────────────────────────────────────────────────────────
    print("🟢 Modo: DOCLING + PDFs REAIS do dataset")
    print(f"   Manual_DPCA_atualizado.pdf ({PDF_MANUAL.stat().st_size//1024} KB)")
    print(f"   Laudo.pdf                  ({PDF_LAUDO.stat().st_size//1024} KB — escaneado, OCR)")

    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions

    # PDF digital — sem OCR (rápido)
    conv_rapido = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=PdfPipelineOptions(do_ocr=False, do_table_structure=True)
            )
        }
    )
    # PDF escaneado — com OCR
    conv_ocr = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=PdfPipelineOptions(do_ocr=True, do_table_structure=True)
            )
        }
    )

    print("\n⏳ Convertendo Manual_DPCA_atualizado.pdf (sem OCR)...")
    md_manual = conv_rapido.convert(str(PDF_MANUAL)).document.export_to_markdown()
    print(f"   ✓ {len(md_manual)} chars extraídos")

    print("⏳ Convertendo Laudo.pdf (com OCR — pode levar minutos)...")
    md_laudo = conv_ocr.convert(str(PDF_LAUDO)).document.export_to_markdown()
    print(f"   ✓ {len(md_laudo)} chars extraídos via OCR")

    corpus_exemplo = {
        "Manual_DPCA":  md_manual,
        "Laudo_Pericial": md_laudo,
    }
else:
    # ────────────────────────────────────────────────────────────
    # OPÇÃO A — Corpus INLINE (default — rápido para a aula)
    # ────────────────────────────────────────────────────────────
    if USE_DOCLING_REAL:
        print("⚠️  USE_DOCLING_REAL=True mas algum PDF do dataset não foi encontrado.")
        print(f"   Esperado em: {DATASET_DIR}")
        print("   Caindo para o corpus INLINE.")
    print("🟡 Modo: Corpus INLINE (rápido) — para usar PDFs reais, mude USE_DOCLING_REAL=True\n")

    PDF_DIR = "/tmp/corpus_juridico"
    os.makedirs(PDF_DIR, exist_ok=True)

    corpus_exemplo = {
        "Lei 11.343": '''
LEI Nº 11.343, DE 23 DE AGOSTO DE 2006
Institui o Sistema Nacional de Políticas Públicas sobre Drogas.

Art. 28. Quem adquirir, guardar, tiver em depósito, transportar ou trouxer consigo,
para consumo pessoal, drogas sem autorização será submetido às seguintes penas:
I - advertência sobre os efeitos das drogas;
II - prestação de serviços à comunidade;
III - medida educativa de comparecimento a programa educativo.

Art. 33. Importar, exportar, remeter, preparar, produzir, fabricar, adquirir, vender,
distribuir, entregar a qualquer título, guardar, ter em depósito, transportar, trazer
consigo, ainda que gratuitamente, sem autorização ou em desacordo com determinação
legal ou regulamentar, matéria-prima, insumo ou produto químico destinado à elaboração
de drogas ilícitas.
Pena - reclusão de 5 a 15 anos e pagamento de 500 a 1.500 salários mínimos.

§ 4º As penas poderão ser reduzidas de um sexto a dois terços, vedada a substituição
por penas restritivas de direitos, desde que o agente colabore efetivamente com a
justiça na investigação e processos penais.
''',
        "Acórdão HC": '''
HABEAS CORPUS Nº 123.456/SP

EMENTA:
Habeas Corpus. Prisão preventiva. Fundamentação adequada. Presença de fumus comissi
delicti e periculum libertatis. Incidência do art. 312 do CPP. Ordem denegada.

RELATÓRIO:
Trata-se de habeas corpus impetrado contra decisão do Tribunal de Justiça do Estado
de São Paulo que manteve a prisão preventiva decretada em desfavor do paciente,
acusado da prática de crime previsto no art. 33 da Lei nº 11.343/2006.

DISPOSITIVO:
Pelo exposto, DENEGO A ORDEM de habeas corpus.
''',
        "Relatório Inteligência": '''
RELATÓRIO DE INTELIGÊNCIA - 1º TRIMESTRE 2025

RESUMO EXECUTIVO:
O presente relatório apresenta análise consolidada das ocorrências registradas no
período de janeiro a março de 2025. Os dados indicam aumento de 12,3% em roubos a
mão armada e redução de 8,7% em furtos.

ANÁLISE POR TIPO DE CRIME:
Homicídio doloso: 446 ocorrências (-5,2%)
Roubo a mão armada: 1.722 ocorrências (+12,3%)
Furto simples: 3.550 ocorrências (-8,7%)
Tráfico de drogas: 260 ocorrências (-14,6%)

OPERAÇÕES ESPECIAIS:
Durante o período, foram deflagradas 18 operações estruturadas. Destaca-se a
Operação "Hércules" de combate ao tráfico de cocaína.
''',
    }

print(f"\n✓ Corpus carregado: {len(corpus_exemplo)} documentos")
for nome, txt in corpus_exemplo.items():
    print(f"   • {nome}: {len(txt)} chars")


In [ ]:
# Chunking jurídico com LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document
import re

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

def chunkar_corpus(corpus):
    """Divide corpus em chunks respeitando estrutura jurídica."""
    separadores = [
        "\n\nArt\.",
        "\n\n§",
        "\nI -",
        "\nII -",
        "\n\n",
        "\n",
        ". ",
        " ",
    ]
    
    splitter = RecursiveCharacterTextSplitter(
        separators=separadores,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
    )
    
    todos_chunks = []
    for nome, texto in corpus.items():
        chunks_texto = splitter.split_text(texto)
        for chunk_id, chunk_texto in enumerate(chunks_texto):
            doc = Document(
                page_content=chunk_texto,
                metadata={
                    'fonte': nome,
                    'tipo_documento': 'juridico',
                    'chunk_id': chunk_id,
                    'chars': len(chunk_texto),
                }
            )
            todos_chunks.append(doc)
    
    return todos_chunks

chunks = chunkar_corpus(corpus_exemplo)
print(f"✅ Chunking completo: {len(chunks)} chunks")

In [ ]:
# Embeddings com BGE-M3 servido pelo Ollama (infra da Aula 1)
# Estratégia: tenta OllamaEmbeddings primeiro; cai para HuggingFaceEmbeddings se o modelo não estiver puxado.
import time

USE_OLLAMA_EMBED = True   # mude para False para forçar HuggingFace

model_embeddings = None
tempo_inicio = time.time()

if USE_OLLAMA_EMBED:
    try:
        from langchain_ollama import OllamaEmbeddings
        print(f'📥 Inicializando OllamaEmbeddings(model="{OLLAMA_EMBED_MODEL}")...')
        model_embeddings = OllamaEmbeddings(
            model=OLLAMA_EMBED_MODEL,            # ollama pull bge-m3
            base_url=OLLAMA_BASE_URL,            # http://localhost:11434
        )
        # Smoke test — gera 1 embedding para validar
        v = model_embeddings.embed_query('teste de embedding jurídico')
        dim_real = len(v)
        print(f'✅ Ollama Embeddings OK | dim={dim_real} | modelo="{OLLAMA_EMBED_MODEL}"')
    except Exception as e:
        print(f'⚠️  Falha ao usar Ollama para embeddings: {e}')
        print(f'   Caindo para HuggingFaceEmbeddings("BAAI/bge-m3")...')
        model_embeddings = None

if model_embeddings is None:
    # Fallback: HuggingFaceEmbeddings (download da Hub na primeira execução, ~2 GB)
    from langchain_community.embeddings import HuggingFaceEmbeddings
    model_embeddings = HuggingFaceEmbeddings(
        model_name='BAAI/bge-m3',
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True},
    )
    dim_real = len(model_embeddings.embed_query('teste'))
    print(f'✅ HuggingFaceEmbeddings OK | dim={dim_real}')

EMBED_DIM = dim_real   # usado para criar o índice OpenSearch/FAISS
tempo = time.time() - tempo_inicio
print(f'⏱️  Tempo de inicialização: {tempo:.2f}s')
print(f'\n📊 Info: dim={EMBED_DIM}, multilíngue (português incluído), multi-granularidade')


In [ ]:
# Gerar embeddings para todos os chunks
print("🔄 Gerando embeddings...")

chunks_textos = [c.page_content for c in chunks]
embeddings = model_embeddings.embed_documents(chunks_textos)

print(f"✅ {len(embeddings)} embeddings gerados")
print(f"   Dimensão: {len(embeddings[0])}")

In [ ]:
# Indexação vetorial — OpenSearch (Aula 1) como padrão; FAISS como fallback automático
import os, requests

USE_OPENSEARCH = True   # mude para False para forçar FAISS local
OPENSEARCH_URL = os.getenv('OPENSEARCH_URL', 'http://localhost:9200')
INDEX_NAME = 'mba-aula2-naive-rag'

vectorstore = None

if USE_OPENSEARCH:
    try:
        requests.get(OPENSEARCH_URL, timeout=3).raise_for_status()
        from langchain_community.vectorstores import OpenSearchVectorSearch

        # Mapping kNN do curso (Aula 1) — ajusta dimensão ao modelo carregado
        print(f'📦 Indexando em OpenSearch ({OPENSEARCH_URL}, índice="{INDEX_NAME}", dim={EMBED_DIM})...')
        vectorstore = OpenSearchVectorSearch.from_documents(
            documents=chunks,
            embedding=model_embeddings,
            opensearch_url=OPENSEARCH_URL,
            index_name=INDEX_NAME,
            engine='faiss',
            space_type='cosinesimil',
            vector_field='embedding',
            text_field='texto',
            bulk_size=200,
        )
        print(f'   ✅ OpenSearch indexou {len(chunks)} chunks (dim={EMBED_DIM})')
    except Exception as e:
        print(f'   ⚠️  OpenSearch indisponível ({e}). Caindo para FAISS local...')
        vectorstore = None

if vectorstore is None:
    from langchain_community.vectorstores import FAISS
    print(f'📦 Criando índice FAISS local (dim={EMBED_DIM})...')
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=model_embeddings,
    )
    print(f'   ✅ FAISS criado com {len(chunks)} vetores')

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5},
)
print(f'\n✅ Retriever pronto | top_k=5 | backend={type(vectorstore).__name__}')


In [ ]:
# Configurar LLM via Ollama (servidor local da Aula 1)
# Caminho A (RECOMENDADO): langchain-ollama nativo
# Caminho B (PORTÁVEL):   ChatOpenAI apontando para o endpoint /v1 do Ollama
#                          (útil para reaproveitar código escrito para OpenAI/vLLM)
import os, requests

print(f'🔌 Configurando LLM via Ollama em {OLLAMA_BASE_URL}...')

USE_OPENAI_COMPAT = False  # True = usa ChatOpenAI(base_url=.../v1); False = ChatOllama nativo

try:
    # Verifica se o Ollama está acessível
    requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5).raise_for_status()

    if USE_OPENAI_COMPAT:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(
            base_url=f'{OLLAMA_BASE_URL}/v1',   # endpoint OpenAI-compatível do Ollama
            api_key='ollama',                    # Ollama ignora; precisa de qualquer string
            model=OLLAMA_LLM_MODEL,              # llama3.2:3b
            temperature=0.1,                     # respostas jurídicas → baixa criatividade
            max_tokens=512,
        )
        LLM_BACKEND = f'ChatOpenAI → Ollama ({OLLAMA_BASE_URL}/v1)'
    else:
        from langchain_ollama import ChatOllama
        llm = ChatOllama(
            model=OLLAMA_LLM_MODEL,              # llama3.2:3b (ou llama3.1:8b se hardware permitir)
            base_url=OLLAMA_BASE_URL,
            temperature=0.1,
            num_predict=512,                     # equivalente a max_tokens
        )
        LLM_BACKEND = f'ChatOllama ({OLLAMA_BASE_URL}, modelo={OLLAMA_LLM_MODEL})'

    # Smoke test
    teste = llm.invoke('Responda em uma palavra: a capital do Brasil é?').content
    print(f'   ✅ LLM OK | backend: {LLM_BACKEND}')
    print(f'   ✅ Smoke test → "{teste[:80].strip()}"')

except Exception as e:
    print(f'   ❌ Falha ao configurar Ollama: {e}')
    print(f'   Verifique:')
    print(f'     1) ollama serve está rodando? curl {OLLAMA_BASE_URL}/api/tags')
    print(f'     2) modelo "{OLLAMA_LLM_MODEL}" foi puxado? ollama pull {OLLAMA_LLM_MODEL}')
    raise


In [ ]:
# Criar prompt template jurídico
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def formatar_contexto(docs):
    """Formata documents em contexto legível com citações."""
    partes = []
    for i, doc in enumerate(docs, 1):
        fonte = doc.metadata.get('fonte', 'N/A')
        bloco = f"[Fonte {i}] {fonte}:\n{doc.page_content}"
        partes.append(bloco)
    return "\n\n".join(partes)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", 
     "Você é um especialista jurídico. Responda APENAS com base no contexto fornecido. "
     "Sempre cite a fonte [Fonte N]. Se não encontrar, diga: 'Não consta nos documentos'.\n\n"
     "CONTEXTO:\n{context}"),
    ("human", "{question}"),
])

# Criar RAG chain (LCEL)
rag_chain = (
    {
        "context": retriever | RunnableLambda(formatar_contexto),
        "question": RunnablePassthrough(),
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ RAG chain criada com LCEL")

In [ ]:
# Testar RAG pipeline
import time

def perguntar(query):
    """Executa query no RAG pipeline com debugging."""
    print(f"\n❓ Pergunta: {query}")
    print("="*80)
    
    # Retrieval
    tempo_inicio = time.time()
    docs = retriever.invoke(query)
    tempo_retrieval = time.time() - tempo_inicio
    
    print(f"\n🔎 [Retrieval] {len(docs)} chunks em {tempo_retrieval*1000:.1f}ms")
    for i, doc in enumerate(docs, 1):
        preview = doc.page_content[:60].replace('\n', ' ')
        print(f"   [{i}] {doc.metadata['fonte']}: \"{preview}...\"")
    
    # Geração
    print(f"\n💬 [Geração]...")
    tempo_inicio = time.time()
    
    try:
        resposta = rag_chain.invoke(query)
        tempo_gen = time.time() - tempo_inicio
        
        print(f"✅ Resposta em {tempo_gen:.2f}s\n")
        print(resposta)
        
    except Exception as e:
        print(f"❌ Erro: {e}")

# Testar
queries = [
    "Qual é a pena para tráfico de drogas?",
    "Quando a prisão preventiva é cabível?",
    "Qual foi o número de homicídios em 2025?",
]

for q in queries:
    perguntar(q)
    print()

In [ ]:
# Persistir o índice para reuso entre sessões / no LAB5 (baseline)
import os

INDEX_DIR = os.path.expanduser('~/mba-rag/aula2_artifacts')
os.makedirs(INDEX_DIR, exist_ok=True)

backend = type(vectorstore).__name__

if backend == 'FAISS':
    caminho = f'{INDEX_DIR}/faiss_index'
    vectorstore.save_local(caminho)
    print(f'✅ Índice FAISS salvo em {caminho}')
    print(f'\n📖 Para carregar em nova sessão:')
    print("     from langchain_community.vectorstores import FAISS")
    print("     from langchain_ollama import OllamaEmbeddings")
    print("     emb = OllamaEmbeddings(model='bge-m3', base_url='http://localhost:11434')")
    print(f"     vs  = FAISS.load_local('{caminho}', emb, allow_dangerous_deserialization=True)")
else:
    # OpenSearch já é persistente — basta reabrir pelo nome do índice
    print(f'✅ Índice OpenSearch "{INDEX_NAME}" já é persistente em {OPENSEARCH_URL}')
    print(f'\n📖 Para reabrir em outra sessão:')
    print("     from langchain_community.vectorstores import OpenSearchVectorSearch")
    print("     from langchain_ollama import OllamaEmbeddings")
    print("     emb = OllamaEmbeddings(model='bge-m3', base_url='http://localhost:11434')")
    print(f"     vs  = OpenSearchVectorSearch(")
    print(f"         opensearch_url='{OPENSEARCH_URL}',")
    print(f"         index_name='{INDEX_NAME}',")
    print(f"         embedding_function=emb,")
    print(f"         vector_field='embedding', text_field='texto')")


In [ ]:
# Exercício: adicionar novo documento e reexecutar uma query
print('''
📚 EXERCÍCIO 1 — Estender o corpus

Passos:
  1. Crie um novo dicionário:
       novo_corpus = {'Sua_Lei': 'texto da sua lei...'}
  2. Use a função `chunkar_corpus(novo_corpus)` para gerar Documents.
  3. Adicione ao índice com `vectorstore.add_documents(novos_chunks)`.
  4. Refaça `perguntar("pergunta sobre o novo conteúdo")`.

📚 EXERCÍCIO 2 — Trocar o LLM

  Se você puxou `llama3.1:8b` (4.9 GB) e tem ≥ 16 GB de RAM:
      from langchain_ollama import ChatOllama
      llm = ChatOllama(model="llama3.1:8b", base_url=OLLAMA_BASE_URL,
                       temperature=0.1, num_predict=512)
  Reconstrua a chain e compare as respostas em latência e qualidade.

📚 EXERCÍCIO 3 — Trocar embedding (medir impacto)

  Reindexe com `nomic-embed-text` (768 dims) e compare a recall:
      OLLAMA_EMBED_MODEL = "nomic-embed-text"
  Reexecute a partir da célula 4. ATENÇÃO: o índice anterior fica inválido —
  apague-o antes (FAISS: rm -rf; OpenSearch: DELETE /<indice>).

📚 EXERCÍCIO 4 — Latência da resposta

  Meça `tempo_retrieval` vs `tempo_gen` para 10 queries seguidas.
  No Ollama, a primeira chamada é mais lenta (cold start do modelo).
  Quanto tempo cai a partir da 2ª? Por quê?
''')
